# Preprocessing

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from ptk import read_F8

# path of stars with fixed metallicity
TRACKS_DIR = "data/ROT_0.00_Z0.02_Y0.284"
# list of features to learn
FEATURES = ["R", "T", "RHO", "MU"]
# number of equidistant points along the radial direction
N_POINTS = 100
# grid of equidistant points
GRID = np.linspace(0, 1, N_POINTS)

---
## Load Profiles and Resample to Fixed Grid
Loads all the F81 tracks of the stars with a fixed metallicity inside the folder `TRACKS_DIR`.\
Then creates a dataset of the raw data resampled along `N_POINTS` equidistant radial points from the center to the outer layer.

In [2]:
%%capture

# directory of tracks
tracks_dir = Path(TRACKS_DIR)

# list of 3d profiles, one for each star: [ (n_timesteps, N_POINTS, n_features), ... ]
X_list = []
# list of stellar mass of each profile: [ (n_timesteps,), ... ]
id_list = []

for F8_file in tracks_dir.rglob("*F81*"):
    F8_str, F8_str_info, F8_track_info = read_F8(F8_file)

    # get mass of the current star's profiles
    mass = F8_track_info["Mass"].item()
    # profiles sorted from center to outer shell and resampled to fixed grid
    profiles_fixed = np.empty( (len(F8_str), N_POINTS, len(FEATURES)), dtype=np.float32 )
    
    for i,profile in enumerate(F8_str):
        # mass radial coordinate without saving it as another array to save memory
        m = np.asarray( profile["M"], dtype=float )
        # sort from 0 to 1 (instead of 1 to 0)
        order = np.argsort(m)
        m = m[order]

        for k,feature in enumerate(FEATURES):
            # radial profile of current feature sorted from center to outer layer
            y = np.asarray(profile[feature], dtype=float)[order]
            # resample to fixed grid
            profiles_fixed[i,:,k] = np.interp(GRID, m, y)
    
    X_list.append( profiles_fixed )
    id_list.append( np.full(len(F8_str), mass, dtype=np.float32) )

X = np.concatenate(X_list) # (total_timesteps, N_POINTS, n_features)
star_id = np.concatenate(id_list) # (total_timesteps,)
del X_list, id_list

In [3]:
print(f"Dataset shape: {X.shape}\nNumber of stars: {len(np.unique(star_id))}")

Dataset shape: (254283, 100, 4)
Number of stars: 88


---
## Log Scaling
The quantities that are always positive and have big numbers are rescaled using log10, while the quantities with zeroes and negative signs are rescaled using arcsinh.

In [4]:
LOG10_FEATURES = ["T", "RHO"]
ARCSINH_FEATURES = ["R"]

for col in LOG10_FEATURES:
    X[:,:,FEATURES.index(col)] = np.log10( X[:,:,FEATURES.index(col)] )
for col in ARCSINH_FEATURES:
    X[:,:,FEATURES.index(col)] = np.arcsinh( X[:,:,FEATURES.index(col)] )

---
## Feature Normalization
Normalize the dataset based on the min and max features so that each value is between 0 and 1.

In [5]:
# get min and max of features
X_min = X.min(axis=(0,1))
X_max = X.max(axis=(0,1))

# scale each feature to [0,1]
X = (X - X_min) / (X_max - X_min)

---
## Train/Test Split
Split in training (80% of masses) and test (20%) datasets.\
Here there's a catch, if I randomly select 80% of the whole dataset of the profiles, the model would already see timesteps close together of a certain star, and the interpolation would be cheated. If I just use the whole evolution of stars in the training phase and other whole stars in the test phase, the model is really used to interpolate stars never seen before.

In [6]:
# select a seed to choose the mass values to use for test phase
rng = np.random.default_rng(81)
# select random mass values
masses = np.unique(star_id)
n_test = int( round(0.2*len(masses)) )
test_masses = rng.choice(masses, size=n_test, replace=False)

# mask for indices of test dataset
is_test = np.isin(star_id, test_masses)
# create training and test datasets
X_train, X_test = X[~is_test], X[is_test]
id_train, id_test = star_id[~is_test], star_id[is_test]
del X

print(f"Stars for training: {len(masses)-n_test}\nStars for test: {n_test}")

Stars for training: 70
Stars for test: 18


## Saving Dataset

In [7]:
np.savez_compressed(
    "data/dataset.npz",
    X_train=X_train, X_test=X_test,
    id_train=id_train, id_test=id_test,
    grid=GRID,
    features=np.array(FEATURES),
    log10_features=np.array(LOG10_FEATURES),
    arcsinh_features=np.array(ARCSINH_FEATURES),
    X_min=X_min, X_max=X_max
)   